# Simple: Impute Missing Values with PAMOLA.CORE

**Goal**: Learn missing value imputation basics in 5 minutes using operation.execute()

**What you'll learn:**
- Load sample data from examples/data_examples/
- Apply statistical imputation using execute()
- Compare before/after results
- Understand data quality improvements

**Prerequisites:**
- Python 3.8+
- PAMOLA.CORE installed
- Basic pandas knowledge

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path
- Verifies all imports work correctly

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── transformations/
        └── 01_impute_missing_values_simple.ipynb  ← You are here
```

In [ ]:
import pandas as pd 
import numpy as np 
import sys 
import os 
import json 
from pathlib import Path 
from datetime import datetime 
from IPython.display import Image, display 
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print(f"   Revision: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.transformations.imputation.impute_missing_values import ImputeMissingValuesOperation
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Sample Data

**How to use:**
- Load data from `examples/data_examples/sample.csv`
- Auto-create sample data if file doesn't exist
- Inspect the dataset structure

**Expected output:** DataFrame with records containing missing values

In [ ]:
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'sample.csv'

if not data_path.exists():
    print("⚠️  File not found, creating sample data...")
    data_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Create sample data with missing values
    np.random.seed(42)
    sample_data = pd.DataFrame({
        'record_id': range(1, 21),
        'age': [25, np.nan, 35, 28, np.nan, 45, 32, np.nan, 29, 41,
               38, 27, np.nan, 33, 36, 42, np.nan, 31, 39, 26],
        'salary': [50000, 65000, np.nan, 55000, 72000, np.nan, 58000, 63000, np.nan, 78000,
                  69000, np.nan, 54000, 67000, 71000, np.nan, 60000, 56000, 75000, np.nan],
        'department': ['Sales', np.nan, 'IT', 'Sales', 'IT', np.nan, 'HR', 'Sales', 'IT', np.nan,
                      'Sales', 'HR', np.nan, 'IT', 'Sales', 'HR', np.nan, 'IT', 'Sales', 'HR'],
        'experience_years': [3, 7, np.nan, 4, 9, 15, np.nan, 8, 5, np.nan,
                           12, 6, 10, np.nan, 11, 14, 7, np.nan, 13, 4]
    })
    sample_data.to_csv(data_path, index=False)
    print(f"✓ Sample data created")

df = read_csv(data_path)
print(f"📊 Dataset loaded: {len(df)} records, {len(df.columns)} columns")
print(f"\n🔍 First 5 records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    missing_count = df[col].isna().sum()
    missing_pct = (missing_count / len(df)) * 100
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:20s} ({dtype_str:10s}): {df[col].nunique()} unique, {missing_count} missing ({missing_pct:.1f}%)")
    else:
        non_null = df[col].dropna()
        if len(non_null) > 0:
            print(f"  {col:20s} ({dtype_str:10s}): min={non_null.min()}, max={non_null.max()}, {missing_count} missing ({missing_pct:.1f}%)")
        else:
            print(f"  {col:20s} ({dtype_str:10s}): all values missing")

## Step 3: Setup Task Environment

**How to use:**
- Create task directory for outputs
- Initialize TaskReporter for tracking
- Setup DataSource with our DataFrame
- Configure progress tracker
- **Configure field strategies** for imputation

**Configuration pattern (production-style):**
```python
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset",
        "field_strategies": {
            "years_of_experience": {"imputation_strategy": "mean"},
            "current_salary_cad": {"imputation_strategy": "median"}
        }
    }
```

In [ ]:
# Configuration helper function (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset",
        "field_strategies": {
            "years_of_experience": {
                "data_type": "numeric",
                "imputation_strategy": "mean"
            },
            "current_salary_cad": {
                "data_type": "numeric",
                "imputation_strategy": "median"
            },
        }
    }

kwargs = create_config_kwargs()
field_strategies = kwargs["field_strategies"]

# Validate fields exist
print(f"\n🔍 Validating field configuration...\n")
for field_name in field_strategies.keys():
    if field_name not in df.columns:
        raise ValueError(
            f"❌ Field '{field_name}' not found in dataset!\n"
            f"Available columns: {', '.join(df.columns)}\n"
            f"Please update 'field_strategies' in create_config_kwargs()"
        )

print(f"✓ Fields to process: {len(field_strategies)}")
for field_name, config in field_strategies.items():
    strategy = config.get('imputation_strategy', 'N/A')
    missing = df[field_name].isna().sum()
    print(f"  {field_name:20s}: {strategy:15s} ({missing} missing values)")

# Create task directory
task_dir = examples_dir / 'data_examples' / 'simple_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="simple_001",
    task_type="transformation",
    description="Imputation of missing values using statistical methods",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

execute_kwargs = {"dataset_name": kwargs["dataset_name"]}
print(f"✓ Config kwargs: {execute_kwargs}")

# Create DataSource
data_source = DataSource(dataframes={"main_dataset": df})
print("✓ DataSource created")

# Setup progress tracker
tracker = HierarchicalProgressTracker(
    total=6,
    description=f"Processing {len(field_strategies)} fields",
    unit="steps",
    track_memory=True,
    level=0,
    bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"
)
print("✓ Progress tracker ready")

print(f"\n✅ Environment setup complete!")

## Step 4: Configure & Execute Operation

**How to use:**
- Create ImputeMissingValuesOperation with full config
- Use `operation.execute()` with explicit execution configs
- Monitor progress through tracker

**Key parameters:**
- `field_strategies`: Dictionary mapping fields to imputation strategies
- `invalid_values`: Optional list of values to treat as missing
- `mode='REPLACE'`: Modify original fields (or 'ENRICH' for new fields)
- `generate_visualization=True`: Create charts
- `save_output=True`: Save to files
- `force_recalculation=False`: Use cache if available

**Available strategies:**
- Numeric: mean, median, mode, min, max, constant, interpolation
- Categorical: mode, most_frequent, constant, random_sample
- Date: mean_date, median_date, mode_date, constant_date, previous_date, next_date
- Text: most_frequent, constant, random_sample

In [ ]:
# Create operation with production-style configuration
operation = ImputeMissingValuesOperation(
    # Core parameters
    field_strategies=field_strategies,
    invalid_values=None,  # Optional: treat specific values as missing
    mode='REPLACE',
    
    # Output configuration
    column_prefix='_',
    output_field_name=None,
    output_format='csv',
    
    # Processing settings
    use_vectorization=False,
    parallel_processes=1,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Operation configured")
print(f"  Fields:           {len(field_strategies)}")
print(f"  Mode:             {operation.mode}")
print(f"  Visualizations:   {operation.generate_visualization}")
print(f"  Save output:      {operation.save_output}")
print(f"  Force recalc:     {operation.force_recalculation}")

# Execute operation
print("\n" + "=" * 80)
print("⚙️  Executing impute missing values...")
print("=" * 80 + "\n")

result = operation.execute(
    data_source=data_source,
    task_dir=task_dir,
    reporter=task_reporter,
    progress_tracker=tracker,
    **execute_kwargs
)

print("\n" + "=" * 80)
print(f"✅ Operation completed!")
print(f"   Status: {result.status}")
print(f"   Artifacts: {len(result.artifacts)}")
print("=" * 80)

## Step 5: Analyze Results

**How to use:**
- Load the processed output from task_dir
- Compare with original data
- Review metrics and artifacts

**Generated artifacts:**
- Processed CSV in output/
- Metrics JSON in metrics/
- Visualizations in visualizations/

In [ ]:
# Display generated artifacts
if result.artifacts:
    print("\n📦 Generated Artifacts:")
    print("=" * 80)
    for artifact in result.artifacts:
        print(f"   [{artifact.category}] {artifact.artifact_type}: {artifact.path.name}")

# Find and load output file
output_files = list(task_dir.glob('output/*.csv'))
if output_files:
    output_file = output_files[0]
    result_df = pd.read_csv(output_file)
    
    print("\n" + "=" * 80)
    print("📊 RESULTS COMPARISON")
    print("=" * 80)
    
    # Show sample records
    print("\n🔍 Sample Processed Records:")
    display_cols = ['record_id', 'age', 'salary', 'department', 'experience_years']
    display_cols = [col for col in display_cols if col in result_df.columns]
    print(result_df[display_cols].head(10))
    
    # Missing values comparison
    print(f"\n📈 Missing Values Comparison:")
    print(f"{'Field':<25} {'Before':>10} {'After':>10} {'Imputed':>10}")
    print("-" * 60)
    for field in field_strategies.keys():
        if field in df.columns and field in result_df.columns:
            before = df[field].isna().sum()
            after = result_df[field].isna().sum()
            imputed = before - after
            print(f"{field:<25} {before:>10} {after:>10} {imputed:>10}")
    
    print("\n" + "=" * 80)
    print("✨ SUMMARY")
    print("=" * 80)
    print(f"  Original records:   {len(df)}")
    print(f"  Processed records:  {len(result_df)}")
    print(f"  Total missing (before): {df.isna().sum().sum()}")
    print(f"  Total missing (after):  {result_df.isna().sum().sum()}")
    
    # Display result metrics
    if result.metrics:
        print("\n📊 Key Metrics:")
        for key, value in list(result.metrics.items())[:10]:
            if isinstance(value, (int, float)):
                if isinstance(value, float):
                    print(f"  {key:30s}: {value:.4f}")
                else:
                    print(f"  {key:30s}: {value}")
    
    print(f"\n💡 All missing values successfully imputed!")
else:
    print("⚠️  No output file found in", task_dir / 'output')

## Step 6: Review Artifacts Location

**How to use:**
- Check all generated files
- Navigate to directories for manual inspection

**Output structure:**
```
examples/data_examples/simple_output/
├── output/           # Processed CSV
├── metrics/          # JSON metrics
├── visualizations/   # PNG charts
└── config/           # Operation config
```

In [ ]:
print("\n📂 OUTPUT DIRECTORY STRUCTURE:")
print("=" * 80)
print(f"\nTask directory: {task_dir.absolute()}\n")

for subdir in ['output', 'metrics', 'visualizations', 'config']:
    path = task_dir / subdir
    if path.exists():
        files = list(path.glob('*'))
        print(f"📁 {subdir}/ ({len(files)} files)")
        for f in files[:5]:
            size_kb = f.stat().st_size / 1024
            print(f"   • {f.name} ({size_kb:.1f} KB)")

print("\n✅ All artifacts saved successfully!")

## Step 7: Detailed Artifact Review

**How to use:**
- Review all generated artifacts in detail
- Automatically loads the NEWEST files from each category
- Excludes system files (like data_types metrics)
- Displays visualizations inline for easy review

**What you'll see:**
1. **Metrics JSON**: Operation performance and effectiveness metrics
2. **Output Data**: Processed records with sample rows
3. **Visualizations**: Charts and graphs from the latest operation run

**Note:** Only the most recent files are shown to avoid confusion from multiple runs

In [ ]:
print("\n📊 DETAILED ARTIFACT REVIEW")
print("=" * 80)

# 1. METRICS JSON (NEWEST - with filtering)
print("\n1️⃣ METRICS (JSON):")
print("-" * 80)

metrics_dir = task_dir / 'metrics'
if not metrics_dir.exists():
    print(f"⚠️  Metrics directory not found: {metrics_dir}")
else:
    metrics_files = list(metrics_dir.glob('*.json'))
    
    if not metrics_files:
        print("⚠️  No metrics files found")
    else:
        # 1) Exclude data_types files
        filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

        if filtered:
            # Use only filtered files
            target_files = filtered
            print(f"✓ Found {len(filtered)} filtered metrics file(s) (excluded data_types)")
        else:
            # Fallback: nothing left after filtering → use all
            target_files = metrics_files
            print(f"⚠ No filtered metrics found → fallback to ALL {len(metrics_files)} file(s)")

        # 2) Pick latest
        target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_metrics_file = target_files[0]
        
        print(f"📄 Loading LATEST: {latest_metrics_file.name}")
        mtime = datetime.fromtimestamp(latest_metrics_file.stat().st_mtime)
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_metrics_file.stat().st_size / 1024:.1f} KB\n")
        print("=" * 80)
        
        try:
            with open(latest_metrics_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # Extract metadata and metrics
            metadata = data.get('metadata', {})
            metrics = data.get('metrics', {})
            
            # METADATA
            print("📋 METADATA:")
            print(f"   Timestamp: {metadata.get('timestamp', 'N/A')}")
            print(f"   Name: {metadata.get('name', 'N/A')}")

            if "operation" in metadata:
                op = metadata["operation"]
                print(f"   Operation: {op.get('class', 'N/A')}.{op.get('function', 'N/A')}")

            # IMPUTED VALUES
            if "imputed_values" in metrics:
                print("\n📊 IMPUTED VALUES:")
                print(f"{'Field':<30} {'Count':>10} {'Percent':>10}")
                print("-" * 55)

                for field, stats in metrics["imputed_values"].items():
                    count = stats.get("count", 0)
                    percent = stats.get("percent", 0) * 100
                    print(f"{field:<30} {count:>10} {percent:>9.2f}%")

            # STATISTICAL COMPARISONS
            if "statistical_comparisons" in metrics:
                print("\n📈 STATISTICAL COMPARISONS:")

                for field, stats in metrics["statistical_comparisons"].items():
                    print(f"\n   {field}:")
                    for stat_name, values in stats.items():
                        before = values.get("before")
                        after = values.get("after")

                        label = stat_name.replace("_", " ").title()

                        if (
                            isinstance(before, (int, float))
                            and isinstance(after, (int, float))
                            and pd.notna(before)
                            and pd.notna(after)
                        ):
                            print(f"      {label:<12}: {before:>10.2f} → {after:>10.2f}")
                        else:
                            print(f"      {label:<12}: N/A")

            # IMPUTATION IMPACTS
            if "imputation_impacts" in metrics:
                print("\n📉 IMPUTATION IMPACTS:")

                for field, stats in metrics["imputation_impacts"].items():
                    print(f"\n   {field}:")
                    for stat_name, values in stats.items():
                        before = values.get("before")
                        after = values.get("after")

                        before_n = values.get("before_sample_count")
                        after_n = values.get("after_sample_count")

                        label = stat_name.replace("_", " ").title()

                        if (
                            isinstance(before, (int, float))
                            and isinstance(after, (int, float))
                            and pd.notna(before)
                            and pd.notna(after)
                        ):
                            print(
                                f"      {label:<20}: "
                                f"{before:>10.2f} → {after:>10.2f}"
                            )
                        else:
                            print(
                                f"      {label:<20}: "
                                f"N/A (samples: {before_n} → {after_n})"
                            )

            # PERFORMANCE
            print("\n⚡ PERFORMANCE:")

            if "execution_time_seconds" in metrics:
                print(f"   Execution Time         : {metrics['execution_time_seconds']:.4f}s")

            if "records_processed" in metrics:
                print(
                    f"   Records Processed     : "
                    f"{metrics['records_processed']} (records affected)"
                )

            if "records_per_second" in metrics:
                print(f"   Records / Second      : {metrics['records_per_second']:.2f}")
        
        except json.JSONDecodeError as e:
            print(f"❌ JSON decode error: {e}")
        except Exception as e:
            print(f"❌ Error reading metrics: {e}")

# 2. OUTPUT DATA (NEWEST)
print("\n\n2️⃣ OUTPUT DATA (First 10 rows):")
print("-" * 80)

output_dir = task_dir / 'output'
if not output_dir.exists():
    print(f"⚠️  Output directory not found: {output_dir}")
else:
    output_files = list(output_dir.glob('*.csv'))
    
    if not output_files:
        print("⚠️  No output files found")
    else:
        # Sort by modification time (newest first)
        output_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_output_file = output_files[0]
        
        # Show file info
        mtime = datetime.fromtimestamp(latest_output_file.stat().st_mtime)
        print(f"✓ Found {len(output_files)} output file(s)")
        print(f"📄 Loading LATEST: {latest_output_file.name}")
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_output_file.stat().st_size / 1024:.1f} KB\n")
        
        try:
            output_df = pd.read_csv(latest_output_file)
            print(f"📊 Shape: {output_df.shape[0]} rows × {output_df.shape[1]} columns")
            print(f"\n{output_df.head(10).to_string()}")
            
            # Show missing value statistics
            print(f"\n\n📊 Missing Values (After Imputation):")
            print("-" * 80)
            print(f"{'Field':<25} {'Missing Count':>15} {'Percentage':>12}")
            print("-" * 55)
            for col in output_df.columns:
                if col != 'record_id':
                    missing = output_df[col].isna().sum()
                    pct = (missing / len(output_df)) * 100
                    print(f"{col:<25} {missing:>15} {pct:>11.2f}%")
        
        except Exception as e:
            print(f"❌ Error reading output: {e}")

# 3. VISUALIZATIONS (NEWEST BATCH)
print("\n\n3️⃣ VISUALIZATIONS:")
print("-" * 80)

viz_dir = task_dir / 'visualizations'
if not viz_dir.exists():
    print(f"⚠️  Visualizations directory not found: {viz_dir}")
else:
    viz_files = list(viz_dir.glob('*.png'))
    
    if not viz_files:
        print("⚠️  No visualizations found")
    else:
        # Sort by modification time (newest first)
        viz_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        
        # Get timestamp from latest file to identify the batch
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            # Group files from same operation (within 10 seconds)
            time_threshold = 10  # seconds
            latest_viz_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < time_threshold
            ]
            
            # Sort batch by name for consistent ordering
            latest_viz_batch.sort(key=lambda x: x.name)
            
            # Show info
            latest_mtime = datetime.fromtimestamp(latest_time)
            print(f"✓ Found {len(viz_files)} total visualization(s)")
            print(f"📄 Showing LATEST batch: {len(latest_viz_batch)} files")
            print(f"   Created: {latest_mtime.strftime('%Y-%m-%d %H:%M:%S')}\n")
            
            # Display each visualization from latest batch
            for i, viz_file in enumerate(latest_viz_batch, 1):
                # Extract readable name from filename
                viz_name = viz_file.stem.replace('_', ' ').title()
                
                print(f"\n{i}. {viz_name}")
                print(f"   File: {viz_file.name}")
                print("-" * 60)
                try:
                    display(Image(filename=str(viz_file)))
                except Exception as e:
                    print(f"   ⚠️  Could not display: {e}")
        
        # Show info about older visualizations if any
        if len(viz_files) > len(latest_viz_batch):
            print(f"\n💡 Note: {len(viz_files) - len(latest_viz_batch)} older visualization(s) not shown")

print("\n\n" + "=" * 80)
print("✅ ARTIFACT REVIEW COMPLETE")
print("=" * 80)

## 🎯 Summary

**What you learned:**  
✅ Load data from examples/data_examples/  
✅ Setup environment with TaskReporter, DataSource, ProgressTracker  
✅ Configure imputation with field-specific strategies  
✅ Execute using operation.execute()  
✅ Analyze results and review artifacts  

**Key takeaways:**
- Different fields can use different imputation strategies
- Mean/median for numeric, mode for categorical
- Statistical imputation preserves data distribution
- Visualizations show before/after comparisons
- All artifacts saved in structured directories

## 📚 Next Steps

**Ready for advanced features?** Try:

📘 **`02_impute_missing_values_advanced.ipynb`** includes:
- Advanced imputation strategies (interpolation, random_sample)
- Custom invalid value handling
- Date and time imputation
- Multiple imputation techniques
- Quality metrics and validation
- Performance optimization for large datasets

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🔧 [API Reference](../../docs/api/)
- 📊 [Data Quality Guide](../../docs/data_quality.md)